In [1]:
import pandas as pd
import numpy as np

FEATURE_END = pd.Timestamp("2024-03-31")
LABEL_START = pd.Timestamp("2024-04-01")
LABEL_END = pd.Timestamp("2024-06-30")

print("Feature window:", "2023-01-01", "to", FEATURE_END.date())
print("Label window:", LABEL_START.date(), "to", LABEL_END.date())

Feature window: 2023-01-01 to 2024-03-31
Label window: 2024-04-01 to 2024-06-30


In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)

print("\nFiles:")
for file in RAW_DIR.iterdir():
    print(" -", file.name)

Project root: c:\Users\shrey\Downloads\FreshBasket_Churn_Project_Structure\FreshBasket_Churn
Raw data directory: c:\Users\shrey\Downloads\FreshBasket_Churn_Project_Structure\FreshBasket_Churn\data\raw

Files:
 - FreshBasket_Loyalty_Churn_Dataset.xlsx


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

EXCEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "FreshBasket_Loyalty_Churn_Dataset.xlsx"
)

customer_profile = pd.read_excel(
    EXCEL_PATH,
    sheet_name="fb Customer Profile",
    header=1
)

monthly_activity = pd.read_excel(
    EXCEL_PATH,
    sheet_name="fb Monthly Activity",
    header=1
)

churn_label = pd.read_excel(
    EXCEL_PATH,
    sheet_name="fb Churn Label",
    header=1
)

print("Customer Profile:", customer_profile.shape)
print("Monthly Activity:", monthly_activity.shape)
print("Churn Label:", churn_label.shape)

Customer Profile: (2608, 10)
Monthly Activity: (28073, 12)
Churn Label: (2605, 6)


In [4]:
print("Customer Profile columns:")
print(customer_profile.columns.tolist())

print("\nMonthly Activity columns:")
print(monthly_activity.columns.tolist())

print("\nChurn Label columns:")
print(churn_label.columns.tolist())

Customer Profile columns:
['CUSTOMER_ID', 'SIGNUP_DATE', 'AGE', 'GENDER', 'CITY', 'MEMBERSHIP_TIER', 'MARKETING_OPT_IN', 'PREFERRED_STORE_ID', 'PREFERRED_CATEGORY', 'HOME_STORE_PRICE_TIER']

Monthly Activity columns:
['CUSTOMER_ID', 'MONTH', 'TRANSACTIONS', 'TOTAL_SPEND', 'AVG_BASKET_VALUE', 'DISTINCT_CATEGORIES', 'PROMO_TXN_PCT', 'APP_SESSIONS', 'EMAILS_OPENED', 'COUPONS_REDEEMED', 'SUPPORT_TICKETS', 'COMPLAINT_FLAG']

Churn Label columns:
['CUSTOMER_ID', 'OBSERVATION_END_DATE', 'LAST_PURCHASE_DATE', 'MONTHS_OBSERVED', 'INSUFFICIENT_HISTORY_FLAG', 'CHURNED']


In [5]:
# ------------------------------------------------------------
# CUSTOMER PROFILE CLEANING
# ------------------------------------------------------------

customer_profile_clean = customer_profile.copy()

customer_profile_clean = (
    customer_profile_clean
    .drop_duplicates(subset="CUSTOMER_ID", keep="first")
    .copy()
)

customer_profile_clean["CITY"] = (
    customer_profile_clean["CITY"]
    .str.strip()
    .str.lower()
    .str.title()
)

customer_profile_clean["MEMBERSHIP_TIER"] = (
    customer_profile_clean["MEMBERSHIP_TIER"]
    .str.strip()
    .str.lower()
    .str.title()
)

customer_profile_clean["GENDER"] = (
    customer_profile_clean["GENDER"]
    .fillna("Unknown")
)

customer_profile_clean["HOME_STORE_PRICE_TIER"] = (
    customer_profile_clean["HOME_STORE_PRICE_TIER"]
    .fillna("Unknown")
)

In [6]:
# ------------------------------------------------------------
# MONTHLY ACTIVITY CLEANING
# ------------------------------------------------------------

monthly_activity_clean = monthly_activity.copy()

# Remove the two conflicting customer-month records
conflicting_rows = [
    (501780, "2024-05-01", -116.18),
    (502065, "2023-04-01", -70.32)
]

for customer_id, month, spend in conflicting_rows:
    mask = (
        (monthly_activity_clean["CUSTOMER_ID"] == customer_id)
        & (
            pd.to_datetime(monthly_activity_clean["MONTH"])
            == pd.Timestamp(month)
        )
        & (monthly_activity_clean["TOTAL_SPEND"] == spend)
    )

    monthly_activity_clean = monthly_activity_clean.loc[~mask]

# Remove exact duplicate rows
monthly_activity_clean = (
    monthly_activity_clean
    .drop_duplicates()
    .copy()
)

# Correct validated negative spend signs
monthly_activity_clean["TOTAL_SPEND"] = (
    monthly_activity_clean["TOTAL_SPEND"].abs()
)

In [7]:
# ------------------------------------------------------------
# CHURN LABEL CLEANING
# ------------------------------------------------------------

churn_label_clean = churn_label.copy()

churn_label_clean = (
    churn_label_clean
    .drop_duplicates(subset="CUSTOMER_ID", keep="first")
    .copy()
)

In [8]:
print("Customer Profile:", customer_profile_clean.shape)
print("Monthly Activity:", monthly_activity_clean.shape)
print("Churn Label:", churn_label_clean.shape)

print("\nUnique customers:")
print(
    "Profile:",
    customer_profile_clean["CUSTOMER_ID"].nunique()
)

print(
    "Activity:",
    monthly_activity_clean["CUSTOMER_ID"].nunique()
)

print(
    "Churn:",
    churn_label_clean["CUSTOMER_ID"].nunique()
)

Customer Profile: (2600, 10)
Monthly Activity: (28013, 12)
Churn Label: (2600, 6)

Unique customers:
Profile: 2600
Activity: 2594
Churn: 2600


In [9]:
# Make sure MONTH is datetime
monthly_activity_clean["MONTH"] = pd.to_datetime(
    monthly_activity_clean["MONTH"]
)

# Keep only information available before the churn-label window
activity_features = monthly_activity_clean[
    monthly_activity_clean["MONTH"] <= FEATURE_END
].copy()

print("Feature activity shape:", activity_features.shape)

print(
    "Feature activity date range:",
    activity_features["MONTH"].min().date(),
    "to",
    activity_features["MONTH"].max().date()
)

Feature activity shape: (23253, 12)
Feature activity date range: 2023-01-01 to 2024-03-01


In [10]:
print(
    "Rows from April-June 2024:",
    activity_features[
        (activity_features["MONTH"] >= LABEL_START) &
        (activity_features["MONTH"] <= LABEL_END)
    ].shape[0]
)

Rows from April-June 2024: 0


In [11]:
print(
    "Maximum feature month:",
    activity_features["MONTH"].max().date()
)

print(
    "Expected maximum feature month:",
    FEATURE_END.date()
)

Maximum feature month: 2024-03-01
Expected maximum feature month: 2024-03-31


In [12]:
print(activity_features.columns.tolist())

['CUSTOMER_ID', 'MONTH', 'TRANSACTIONS', 'TOTAL_SPEND', 'AVG_BASKET_VALUE', 'DISTINCT_CATEGORIES', 'PROMO_TXN_PCT', 'APP_SESSIONS', 'EMAILS_OPENED', 'COUPONS_REDEEMED', 'SUPPORT_TICKETS', 'COMPLAINT_FLAG']


In [13]:
rfm_features = (
    activity_features
    .groupby("CUSTOMER_ID")
    .agg(
        total_transactions=("TRANSACTIONS", "sum"),
        total_spend=("TOTAL_SPEND", "sum"),
        avg_basket_value=("AVG_BASKET_VALUE", "mean"),
        active_months=("TRANSACTIONS", lambda x: (x > 0).sum()),
        observed_months=("MONTH", "nunique")
    )
    .reset_index()
)

In [14]:
last_purchase = (
    activity_features[
        activity_features["TRANSACTIONS"] > 0
    ]
    .groupby("CUSTOMER_ID")["MONTH"]
    .max()
    .reset_index()
    .rename(columns={"MONTH": "LAST_PURCHASE_MONTH"})
)

In [17]:
# Start with the sentinel value for customers
# who have no purchase in the feature window
rfm_features["RECENCY_MONTHS"] = 999

# Identify customers who have at least one historical purchase
has_purchase = rfm_features["LAST_PURCHASE_MONTH"].notna()

# Calculate recency only for those customers
rfm_features.loc[has_purchase, "RECENCY_MONTHS"] = (
    FEATURE_END.to_period("M")
    - rfm_features.loc[has_purchase, "LAST_PURCHASE_MONTH"].dt.to_period("M")
).apply(lambda x: x.n)

In [18]:
print(
    "Customers with no historical purchase:",
    (~has_purchase).sum()
)

print(
    "Recency range:",
    rfm_features["RECENCY_MONTHS"].min(),
    "to",
    rfm_features["RECENCY_MONTHS"].max()
)

display(rfm_features.head(10))

Customers with no historical purchase: 113
Recency range: 0 to 999


,CUSTOMER_ID,total_transactions,total_spend,avg_basket_value,active_months,observed_months,LAST_PURCHASE_MONTH,RECENCY_MONTHS
0,500001,68,1512.80,22.485714,14,14,2024-03-01,0
1,500002,53,1792.40,33.348889,9,9,2024-03-01,0
2,500003,69,2197.55,31.824286,14,14,2024-03-01,0
3,500004,75,1778.75,23.412857,14,14,2024-03-01,0
4,500005,0,0.00,0.000000,0,1,NaT,999
5,500006,37,1212.45,30.600667,14,15,2024-02-01,1
6,500007,52,1925.44,37.458000,15,15,2024-03-01,0
7,500008,97,3915.59,40.338000,15,15,2024-03-01,0
8,500009,49,807.99,16.413333,15,15,2024-03-01,0
9,500010,16,855.03,36.764000,10,15,2024-02-01,1


In [19]:
rfm_features["NO_HISTORICAL_PURCHASE"] = (
    rfm_features["LAST_PURCHASE_MONTH"].isna().astype(int)
)

display(
    rfm_features[
        [
            "CUSTOMER_ID",
            "LAST_PURCHASE_MONTH",
            "RECENCY_MONTHS",
            "NO_HISTORICAL_PURCHASE"
        ]
    ].head(10)
)

,CUSTOMER_ID,LAST_PURCHASE_MONTH,RECENCY_MONTHS,NO_HISTORICAL_PURCHASE
0,500001,2024-03-01,0,0
1,500002,2024-03-01,0,0
2,500003,2024-03-01,0,0
3,500004,2024-03-01,0,0
4,500005,NaT,999,1
5,500006,2024-02-01,1,0
6,500007,2024-03-01,0,0
7,500008,2024-03-01,0,0
8,500009,2024-03-01,0,0
9,500010,2024-02-01,1,0


In [20]:
print("RFM feature shape:", rfm_features.shape)

print("\nMissing values:")
print(
    rfm_features[
        [
            "total_transactions",
            "total_spend",
            "avg_basket_value",
            "active_months",
            "observed_months",
            "RECENCY_MONTHS",
            "NO_HISTORICAL_PURCHASE"
        ]
    ].isna().sum()
)

print("\nSummary:")
display(
    rfm_features[
        [
            "total_transactions",
            "total_spend",
            "avg_basket_value",
            "active_months",
            "observed_months",
            "RECENCY_MONTHS"
        ]
    ].describe()
)

RFM feature shape: (2481, 9)

Missing values:
total_transactions        0
total_spend               0
avg_basket_value          0
active_months             0
observed_months           0
RECENCY_MONTHS            0
NO_HISTORICAL_PURCHASE    0
dtype: int64

Summary:


,total_transactions,total_spend,avg_basket_value,active_months,observed_months,RECENCY_MONTHS
count,2481.000000,2481.000000,2481.000000,2481.000000,2481.000000,2481.000000
mean,38.675131,1250.581859,27.742550,8.569931,9.372430,47.819428
std,31.731689,1096.178797,10.646888,5.153222,4.988773,207.864345
min,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,11.000000,319.940000,22.235714,4.000000,5.000000,0.000000
50%,33.000000,1007.350000,28.676429,9.000000,10.000000,0.000000
75%,61.000000,1932.750000,34.569231,14.000000,14.000000,5.000000
max,165.000000,7841.166790,59.859333,15.000000,15.000000,999.000000


In [21]:
engagement_features = (
    activity_features
    .groupby("CUSTOMER_ID")
    .agg(
        avg_app_sessions=("APP_SESSIONS", "mean"),
        total_app_sessions=("APP_SESSIONS", "sum"),
        avg_emails_opened=("EMAILS_OPENED", "mean"),
        total_emails_opened=("EMAILS_OPENED", "sum"),
        total_coupons_redeemed=("COUPONS_REDEEMED", "sum"),
        avg_distinct_categories=("DISTINCT_CATEGORIES", "mean"),
        avg_promo_txn_pct=("PROMO_TXN_PCT", "mean")
    )
    .reset_index()
)

display(engagement_features.head())

,CUSTOMER_ID,avg_app_sessions,total_app_sessions,avg_emails_opened,total_emails_opened,total_coupons_redeemed,avg_distinct_categories,avg_promo_txn_pct
0,500001,2.142857,30,1.214286,17,7,2.857143,0.197143
1,500002,2.333333,21,0.333333,3,5,2.000000,0.307778
2,500003,3.285714,46,0.428571,6,10,2.428571,0.342143
3,500004,3.928571,55,1.428571,20,9,3.857143,0.243571
4,500005,0.000000,0,0.000000,0,0,1.000000,0.000000


In [22]:
print("Engagement feature shape:", engagement_features.shape)

print("\nMissing values:")
print(engagement_features.isna().sum())

print("\nSummary:")
display(
    engagement_features.drop(columns=["CUSTOMER_ID"]).describe()
)

Engagement feature shape: (2481, 8)

Missing values:
CUSTOMER_ID                0
avg_app_sessions           0
total_app_sessions         0
avg_emails_opened          0
total_emails_opened        0
total_coupons_redeemed     0
avg_distinct_categories    0
avg_promo_txn_pct          0
dtype: int64

Summary:


,avg_app_sessions,total_app_sessions,avg_emails_opened,total_emails_opened,total_coupons_redeemed,avg_distinct_categories,avg_promo_txn_pct
count,2481.000000,2481.000000,2481.000000,2481.000000,2481.000000,2481.000000,2481.000000
mean,1.793788,18.953245,0.925545,9.478033,5.245466,2.283127,0.253345
std,1.219456,16.481277,0.683891,8.566699,3.992408,0.990032,0.095055
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.833333,4.000000,0.333333,2.000000,2.000000,1.615385,0.214667
50%,1.714286,15.000000,0.875000,6.000000,5.000000,2.230769,0.263333
75%,2.625000,30.000000,1.466667,17.000000,8.000000,2.928571,0.306000
max,11.000000,75.000000,6.000000,39.000000,18.000000,7.500000,0.770000


In [23]:
support_features = (
    activity_features
    .groupby("CUSTOMER_ID")
    .agg(
        total_support_tickets=("SUPPORT_TICKETS", "sum"),
        avg_support_tickets=("SUPPORT_TICKETS", "mean"),
        complaint_count=("COMPLAINT_FLAG", "sum"),
        complaint_rate=("COMPLAINT_FLAG", "mean")
    )
    .reset_index()
)

support_features["has_complaint"] = (
    support_features["complaint_count"] > 0
).astype(int)

display(support_features.head())

,CUSTOMER_ID,total_support_tickets,avg_support_tickets,complaint_count,complaint_rate,has_complaint
0,500001,3,0.214286,1,0.071429,1
1,500002,1,0.111111,0,0.000000,0
2,500003,2,0.142857,1,0.071429,1
3,500004,3,0.214286,1,0.071429,1
4,500005,0,0.000000,0,0.000000,0


In [24]:
print("Support feature shape:", support_features.shape)

print("\nMissing values:")
print(support_features.isna().sum())

print("\nSummary:")
display(
    support_features.drop(columns=["CUSTOMER_ID"]).describe()
)

Support feature shape: (2481, 6)

Missing values:
CUSTOMER_ID              0
total_support_tickets    0
avg_support_tickets      0
complaint_count          0
complaint_rate           0
has_complaint            0
dtype: int64

Summary:


,total_support_tickets,avg_support_tickets,complaint_count,complaint_rate,has_complaint
count,2481.000000,2481.000000,2481.000000,2481.000000,2481.000000
mean,1.471987,0.199371,0.398227,0.054529,0.331721
std,1.354250,0.277710,0.624291,0.125503,0.470926
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.133333,0.000000,0.000000,0.000000
75%,2.000000,0.250000,1.000000,0.071429,1.000000
max,8.000000,3.000000,4.000000,1.000000,1.000000


In [25]:
modeling_df = customer_profile_clean.copy()

print("Starting customers:", modeling_df.shape)

Starting customers: (2600, 10)


In [26]:
modeling_df = modeling_df.merge(
    rfm_features,
    on="CUSTOMER_ID",
    how="left"
)

modeling_df = modeling_df.merge(
    engagement_features,
    on="CUSTOMER_ID",
    how="left"
)

modeling_df = modeling_df.merge(
    support_features,
    on="CUSTOMER_ID",
    how="left"
)

print("Shape after feature merges:", modeling_df.shape)
print("Unique customers:", modeling_df["CUSTOMER_ID"].nunique())

Shape after feature merges: (2600, 30)
Unique customers: 2600


In [27]:
modeling_df = modeling_df.merge(
    churn_label_clean[
        ["CUSTOMER_ID", "CHURNED"]
    ],
    on="CUSTOMER_ID",
    how="left"
)

print("Final rows:", len(modeling_df))
print("Unique customers:", modeling_df["CUSTOMER_ID"].nunique())
print("Missing target:", modeling_df["CHURNED"].isna().sum())

print("\nTarget distribution:")
print(modeling_df["CHURNED"].value_counts())

Final rows: 2600
Unique customers: 2600
Missing target: 0

Target distribution:
CHURNED
0    1782
1     818
Name: count, dtype: int64


In [28]:
no_prelabel_activity = modeling_df["observed_months"].isna()

print(
    "Customers with no pre-label activity:",
    no_prelabel_activity.sum()
)

print("\nChurn distribution:")
print(
    modeling_df.loc[
        no_prelabel_activity,
        "CHURNED"
    ].value_counts()
)

Customers with no pre-label activity: 119

Churn distribution:
CHURNED
0    117
1      2
Name: count, dtype: int64


In [29]:
modeling_df["NO_PRELABEL_ACTIVITY"] = (
    modeling_df["observed_months"].isna()
).astype(int)

print(
    modeling_df["NO_PRELABEL_ACTIVITY"].value_counts()
)

NO_PRELABEL_ACTIVITY
0    2481
1     119
Name: count, dtype: int64


In [30]:
activity_feature_cols = [
    "total_transactions",
    "total_spend",
    "avg_basket_value",
    "active_months",
    "observed_months",
    "avg_app_sessions",
    "total_app_sessions",
    "avg_emails_opened",
    "total_emails_opened",
    "total_coupons_redeemed",
    "avg_distinct_categories",
    "avg_promo_txn_pct",
    "total_support_tickets",
    "avg_support_tickets",
    "complaint_count",
    "complaint_rate",
    "has_complaint"
]

modeling_df.loc[
    modeling_df["NO_PRELABEL_ACTIVITY"] == 1,
    activity_feature_cols
] = 0

In [31]:
print("Final modeling dataset shape:", modeling_df.shape)
print("Unique customers:", modeling_df["CUSTOMER_ID"].nunique())

print("\nMissing values:")
display(
    modeling_df.isna().sum().sort_values(ascending=False).head(15)
)

print("\nNo-prelabel-activity check:")
display(
    modeling_df[
        modeling_df["NO_PRELABEL_ACTIVITY"] == 1
    ][
        [
            "CUSTOMER_ID",
            "total_transactions",
            "total_spend",
            "active_months",
            "observed_months",
            "NO_PRELABEL_ACTIVITY",
            "CHURNED"
        ]
    ].head(10)
)

Final modeling dataset shape: (2600, 32)
Unique customers: 2600

Missing values:


LAST_PURCHASE_MONTH       232
RECENCY_MONTHS            119
NO_HISTORICAL_PURCHASE    119
AGE                        78
CITY                        0
MEMBERSHIP_TIER             0
SIGNUP_DATE                 0
CUSTOMER_ID                 0
PREFERRED_STORE_ID          0
PREFERRED_CATEGORY          0
total_transactions          0
HOME_STORE_PRICE_TIER       0
total_spend                 0
avg_basket_value            0
MARKETING_OPT_IN            0
dtype: int64


No-prelabel-activity check:


,CUSTOMER_ID,total_transactions,total_spend,active_months,observed_months,NO_PRELABEL_ACTIVITY,CHURNED
10,500011,0.0,0.0,0.0,0.0,1,0
52,500053,0.0,0.0,0.0,0.0,1,0
60,500061,0.0,0.0,0.0,0.0,1,0
62,500063,0.0,0.0,0.0,0.0,1,0
86,500087,0.0,0.0,0.0,0.0,1,0
101,500102,0.0,0.0,0.0,0.0,1,0
113,500114,0.0,0.0,0.0,0.0,1,0
117,500118,0.0,0.0,0.0,0.0,1,0
149,500150,0.0,0.0,0.0,0.0,1,0
161,500162,0.0,0.0,0.0,0.0,1,0


In [32]:
print("All modelling columns:")
for col in modeling_df.columns:
    print("-", col)

All modelling columns:
- CUSTOMER_ID
- SIGNUP_DATE
- AGE
- GENDER
- CITY
- MEMBERSHIP_TIER
- MARKETING_OPT_IN
- PREFERRED_STORE_ID
- PREFERRED_CATEGORY
- HOME_STORE_PRICE_TIER
- total_transactions
- total_spend
- avg_basket_value
- active_months
- observed_months
- LAST_PURCHASE_MONTH
- RECENCY_MONTHS
- NO_HISTORICAL_PURCHASE
- avg_app_sessions
- total_app_sessions
- avg_emails_opened
- total_emails_opened
- total_coupons_redeemed
- avg_distinct_categories
- avg_promo_txn_pct
- total_support_tickets
- avg_support_tickets
- complaint_count
- complaint_rate
- has_complaint
- CHURNED
- NO_PRELABEL_ACTIVITY


In [33]:
modeling_df["SIGNUP_DATE"] = pd.to_datetime(
    modeling_df["SIGNUP_DATE"]
)

modeling_df["SIGNUP_YEAR"] = (
    modeling_df["SIGNUP_DATE"].dt.year
)

modeling_df["SIGNUP_MONTH"] = (
    modeling_df["SIGNUP_DATE"].dt.month
)

modeling_df["CUSTOMER_TENURE_MONTHS"] = (
    (FEATURE_END.year - modeling_df["SIGNUP_DATE"].dt.year) * 12
    + (FEATURE_END.month - modeling_df["SIGNUP_DATE"].dt.month)
)

display(
    modeling_df[
        [
            "CUSTOMER_ID",
            "SIGNUP_DATE",
            "SIGNUP_YEAR",
            "SIGNUP_MONTH",
            "CUSTOMER_TENURE_MONTHS"
        ]
    ].head()
)

,CUSTOMER_ID,SIGNUP_DATE,SIGNUP_YEAR,SIGNUP_MONTH,CUSTOMER_TENURE_MONTHS
0,500001,2022-08-17,2022,8,19
1,500002,2023-06-24,2023,6,9
2,500003,2021-03-31,2021,3,36
3,500004,2021-07-08,2021,7,32
4,500005,2023-09-28,2023,9,6


In [34]:
print("=" * 60)
print("FINAL LEAKAGE AUDIT")
print("=" * 60)

print("\nFeature window:")
print("Start:", activity_features["MONTH"].min())
print("End:", activity_features["MONTH"].max())

print("\nLabel window:")
print("Start:", LABEL_START)
print("End:", LABEL_END)

print("\nApril-June records accidentally included:")
leakage_rows = activity_features[
    (activity_features["MONTH"] >= LABEL_START) &
    (activity_features["MONTH"] <= LABEL_END)
]

print(len(leakage_rows))

print("\nPotential leakage columns excluded from modelling:")
print([
    "LAST_PURCHASE_DATE",
    "MONTHS_OBSERVED",
    "INSUFFICIENT_HISTORY_FLAG",
    "LAST_PURCHASE_MONTH"
])

print("\nTarget:")
print("CHURNED")

print("\nCustomer count:")
print(modeling_df["CUSTOMER_ID"].nunique())

FINAL LEAKAGE AUDIT

Feature window:
Start: 2023-01-01 00:00:00
End: 2024-03-01 00:00:00

Label window:
Start: 2024-04-01 00:00:00
End: 2024-06-30 00:00:00

April-June records accidentally included:
0

Potential leakage columns excluded from modelling:
['LAST_PURCHASE_DATE', 'MONTHS_OBSERVED', 'INSUFFICIENT_HISTORY_FLAG', 'LAST_PURCHASE_MONTH']

Target:
CHURNED

Customer count:
2600


In [35]:
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "customer_modeling_dataset.csv"
)

modeling_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved modelling dataset to:")
print(OUTPUT_PATH)

Saved modelling dataset to:
c:\Users\shrey\Downloads\FreshBasket_Churn_Project_Structure\FreshBasket_Churn\data\processed\customer_modeling_dataset.csv


In [ ]:
check_df = pd.read_csv(OUTPUT_PATH)

print("Saved shape:", check_df.shape)
print("Unique customers:", check_df["CUSTOMER_ID"].nunique())
print("Target missing:", check_df["CHURNED"].isna().sum())

Saved shape: (2600, 35)
Unique customers: 2600
Target missing: 0


: 